# FcaDreamUHD — Training Kaggle (2× T4)

## Due stage, come gli autori originali:
| Stage | Script | Config | Cosa traina |
|---|---|---|---|
| **Stage 1** | `basicsr/train.py` | `VAE_LL_kaggle.yml` | FE-VAE con FCA da zero |
| **Stage 2** | `basicsr/train.py` | `DreamUHD_LL_kaggle.yml` | DreamUHD completo, carica VAE da Stage 1 |

**Accelerator**: GPU → T4 × 2

## 0. Verifica GPU

In [ ]:
import torch
print(f"GPU disponibili: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  —  {p.total_memory / 1024**3:.1f} GB VRAM")

## 1. Dipendenze

In [ ]:
%%bash
pip install -q basicsr facexlib lpips einops timm scikit-image pyyaml tensorboard

## 2. Percorsi — **modifica solo questi**

In [ ]:
import os, sys, shutil, yaml, glob, subprocess

# ─── MODIFICA QUESTI PATH ────────────────────────────────────────────────────
CODE_SRC  = "/kaggle/input/<nome-dataset-codice>/FcaDreamuhd"
DATA_ROOT = "/kaggle/input/<nome-dataset-immagini>"   # contiene training_set/ e testing_set/
# ─────────────────────────────────────────────────────────────────────────────

CODE_DEST = "/kaggle/working/FcaDreamuhd"
EXP_DIR   = "/kaggle/working/experiments"   # ← qui vengono salvati i checkpoint

if os.path.exists(CODE_DEST):
    shutil.rmtree(CODE_DEST)
shutil.copytree(CODE_SRC, CODE_DEST)
os.makedirs(EXP_DIR, exist_ok=True)
print(f"✅ Codice copiato in: {CODE_DEST}")
print(f"✅ Esperimenti in   : {EXP_DIR}")

## 3. Genera il `root_file.txt` per il VAE

In [ ]:
# Struttura attesa:
# DATA_ROOT/
#   training_set/gt/    ← immagini GT training (VAE Stage 1 + Stage 2)
#   training_set/input/ ← immagini LQ training (solo Stage 2)
#   testing_set/gt/     ← immagini GT validation
#   testing_set/input/  ← immagini LQ validation (solo Stage 2)

GT_TRAIN = os.path.join(DATA_ROOT, "training_set/gt")
GT_VAL   = os.path.join(DATA_ROOT, "testing_set/gt")

ROOT_FILE_TRAIN = "/kaggle/working/vae_train_paths.txt"
exts = ('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')
train_paths = sorted([p for p in glob.glob(f"{GT_TRAIN}/**/*", recursive=True)
                      if p.lower().endswith(exts)])
with open(ROOT_FILE_TRAIN, 'w') as f:
    f.write('\n'.join(train_paths))

print(f"✅ root_file: {len(train_paths)} immagini GT → {ROOT_FILE_TRAIN}")
print(f"   Esempio  : {train_paths[0] if train_paths else 'ERRORE: nessuna immagine trovata!'}")

## 4. Config Stage 1 — VAE

In [ ]:
VAE_YML_SRC  = os.path.join(CODE_DEST, "options/VAE_LL.yml")
VAE_YML_DEST = "/kaggle/working/VAE_LL_kaggle.yml"

VAE_EXP_DIR = os.path.join(EXP_DIR, "VAE_LL")
os.makedirs(VAE_EXP_DIR, exist_ok=True)

with open(VAE_YML_SRC, "r") as f:
    vae_cfg = yaml.safe_load(f)

# Dataset
vae_cfg["datasets"]["train"]["root_file"] = ROOT_FILE_TRAIN
vae_cfg["datasets"]["train"]["rand_num"]  = 0
vae_cfg["datasets"]["val"]["dataroot_lq"] = GT_VAL

# ⚠️ Forza i path esperimenti in /kaggle/working/
# (basicsr altrimenti usa un path hardcoded dal cluster originale)
vae_cfg["path"]["experiments_root"] = VAE_EXP_DIR
vae_cfg["path"]["models"]           = os.path.join(VAE_EXP_DIR, "models")
vae_cfg["path"]["training_states"]  = os.path.join(VAE_EXP_DIR, "training_states")
vae_cfg["path"]["log"]              = VAE_EXP_DIR
vae_cfg["path"]["visualization"]    = os.path.join(VAE_EXP_DIR, "visualization")

# Nessun resume (training da zero)
vae_cfg["path"]["resume_state"] = None
# Per riprendere da sessione precedente, decommenta:
# vae_cfg["path"]["resume_state"] = f"{VAE_EXP_DIR}/training_states/60000.state"

with open(VAE_YML_DEST, "w") as f:
    yaml.dump(vae_cfg, f, default_flow_style=False, allow_unicode=True)

print(f"✅ VAE config: {VAE_YML_DEST}")
print(f"   experiments_root : {vae_cfg['path']['experiments_root']}")
print(f"   training_states  : {vae_cfg['path']['training_states']}")
print(f"   res_type         : {vae_cfg['network_g']['ddconfig']['res_type']}")
print(f"   batch/GPU        : {vae_cfg['datasets']['train']['batch_size_per_gpu']}")
print(f"   total_iter       : {vae_cfg['train']['total_iter']}")
print(f"   resume_state     : {vae_cfg['path']['resume_state']}")

## 5. STAGE 1 — Addestra il FE-VAE da zero

```bash
# Equivalente a:
python basicsr/train.py -opt options/VAE_LL.yml
```

> **Stima tempo**: 122.000 iter × ~0.5s/iter su 2×T4 ≈ **~17h** → 2 sessioni Kaggle  
> Checkpoint ogni 1.000 iter (`net_g_latest`) e ogni 5.000 iter (numerato) in `/kaggle/working/experiments/VAE_LL/`

In [ ]:
TRAIN_PY = os.path.join(CODE_DEST, "basicsr/train.py")

cmd = [
    "torchrun",
    "--nproc_per_node=2",
    "--master_port=29500",
    TRAIN_PY,
    "-opt", VAE_YML_DEST,
    "--launcher", "pytorch",
]
print("[STAGE 1] Comando:", " ".join(cmd))
print("─" * 70)

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
    env={**os.environ, "PYTHONPATH": CODE_DEST},
)
for line in proc.stdout:
    print(line, end="", flush=True)

proc.wait()
print(f"\nExit code: {proc.returncode}")

## 5b. ⚡ Copia checkpoint (esegui in qualsiasi momento mentre il training gira)

Se il path non è stato fixato correttamente, usa questa cella per copiare manualmente i checkpoint in `/kaggle/working/`.

In [ ]:
# Cerca il checkpoint VAE ovunque e copialo in /kaggle/working/
import glob as _glob

# Cerca in tutti i path possibili
search_patterns = [
    "/kaggle/working/experiments/VAE_LL/models/*.pth",
    "/mnt/**/VAE_LL/models/*.pth",
    "/root/**/VAE_LL/models/*.pth",
]
found = []
for pattern in search_patterns:
    found += _glob.glob(pattern, recursive=True)

if found:
    DST = "/kaggle/working/vae_checkpoints"
    os.makedirs(DST, exist_ok=True)
    for f in found:
        shutil.copy(f, DST)
        print(f"✅ Copiato: {f} → {DST}")
else:
    print("❌ Nessun checkpoint trovato. Il training sta ancora girando?")
    # Mostra dove stanno girando gli esperimenti
    !find / -name 'net_g_latest.pth' 2>/dev/null | head -5

## 6. Verifica VAE trainato

In [ ]:
# Cerca il peso VAE in tutti i path possibili
candidates = [
    f"{VAE_EXP_DIR}/models/net_g_latest.pth",
    "/kaggle/working/vae_checkpoints/net_g_latest.pth",
]
VAE_TRAINED = next((p for p in candidates if os.path.exists(p)), None)

if VAE_TRAINED is None:
    raise FileNotFoundError(
        "VAE non trovato. Esegui la cella 5b per cercarlo e copiarlo."
    )
print(f"✅ VAE trainato trovato: {VAE_TRAINED}")

## 7. Config Stage 2 — DreamUHD + FCA

In [ ]:
DREAM_YML_SRC  = os.path.join(CODE_DEST, "options/DreamUHD_LL.yml")
DREAM_YML_DEST = "/kaggle/working/DreamUHD_LL_kaggle.yml"

with open(DREAM_YML_SRC, "r") as f:
    dream_cfg = yaml.safe_load(f)

DREAM_NAME    = dream_cfg.get("name", "FcaDreamUHD")
DREAM_EXP_DIR = os.path.join(EXP_DIR, DREAM_NAME)
os.makedirs(DREAM_EXP_DIR, exist_ok=True)

# Dataset
dream_cfg["datasets"]["train"]["dataroot_gt"] = os.path.join(DATA_ROOT, "training_set/gt")
dream_cfg["datasets"]["train"]["dataroot_lq"] = os.path.join(DATA_ROOT, "training_set/input")
dream_cfg["datasets"]["val"]["dataroot_gt"]   = os.path.join(DATA_ROOT, "testing_set/gt")
dream_cfg["datasets"]["val"]["dataroot_lq"]   = os.path.join(DATA_ROOT, "testing_set/input")

# VAE prodotto nello Stage 1
dream_cfg["network_g"]["vae_weight"] = VAE_TRAINED
dream_cfg["network_g"]["config"]     = os.path.join(CODE_DEST, "options/VAE_LL.yml")

# ⚠️ Forza i path esperimenti in /kaggle/working/
dream_cfg["path"]["experiments_root"] = DREAM_EXP_DIR
dream_cfg["path"]["models"]           = os.path.join(DREAM_EXP_DIR, "models")
dream_cfg["path"]["training_states"]  = os.path.join(DREAM_EXP_DIR, "training_states")
dream_cfg["path"]["log"]              = DREAM_EXP_DIR
dream_cfg["path"]["visualization"]    = os.path.join(DREAM_EXP_DIR, "visualization")

# Nessun pretrain — da zero come gli autori
dream_cfg["path"]["pretrain_network_g"] = None
dream_cfg["path"]["pretrain_network_d"] = None
dream_cfg["path"]["resume_state"]       = None
# Per resume Stage 2:
# dream_cfg["path"]["resume_state"] = f"{DREAM_EXP_DIR}/training_states/25000.state"

with open(DREAM_YML_DEST, "w") as f:
    yaml.dump(dream_cfg, f, default_flow_style=False, allow_unicode=True)

print(f"✅ DreamUHD config: {DREAM_YML_DEST}")
print(f"   experiments_root : {dream_cfg['path']['experiments_root']}")
print(f"   vae_weight       : {dream_cfg['network_g']['vae_weight']}")
print(f"   resume_state     : {dream_cfg['path']['resume_state']}")

## 8. STAGE 2 — Addestra DreamUHD + FCA da zero

```bash
# Equivalente a:
python basicsr/train.py -opt options/DreamUHD_LL.yml
```

In [ ]:
cmd = [
    "torchrun",
    "--nproc_per_node=2",
    "--master_port=29500",
    TRAIN_PY,
    "-opt", DREAM_YML_DEST,
    "--launcher", "pytorch",
]
print("[STAGE 2] Comando:", " ".join(cmd))
print("─" * 70)

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
    env={**os.environ, "PYTHONPATH": CODE_DEST},
)
for line in proc.stdout:
    print(line, end="", flush=True)

proc.wait()
print(f"\nExit code: {proc.returncode}")

## 9. Salva i checkpoint finali

In [ ]:
OUT_DIR = "/kaggle/working/output_weights"
os.makedirs(OUT_DIR, exist_ok=True)

for pattern in [
    f"{VAE_EXP_DIR}/models/*.pth",
    f"{DREAM_EXP_DIR}/models/*.pth",
]:
    for ckpt in glob.glob(pattern):
        shutil.copy(ckpt, OUT_DIR)
        print(f"Salvato: {os.path.basename(ckpt)}")

print(f"\n✅ Pesi in: {OUT_DIR}")

---
## Piano sessioni Kaggle (9h max per sessione)

| Sessione | Stage | Iter | Tempo stimato |
|---|---|---|---|
| 1 | VAE (Stage 1) | 0 → ~35k | ~9h |
| 2 | VAE (Stage 1) resume | 35k → 122k | ~9h |
| 3 | DreamUHD (Stage 2) | 0 → ~30k | ~9h |
| 4 | DreamUHD (Stage 2) resume | 30k → 61k | ~9h |

**Per il resume**: decommenta la riga `resume_state` nella cella del config corrispondente e punta all'ultimo `.state` in `training_states/`.  
**I checkpoint sono in**: `/kaggle/working/experiments/VAE_LL/training_states/`